In [1]:

# Cell 1 - Clone repo and install dependencies
!git clone https://github.com/JormayBusso/pixels-to-macros.git
%cd pixels-to-macros

!git fetch --all
!git reset --hard origin/dev

!pip install -q --upgrade pip
!pip install -q segmentation-models-pytorch timm datasets transformers albumentations opencv-python-headless safetensors tqdm Pillow
!grep -vE '^torch|^torchvision|coremltools' training/requirements.txt > /tmp/kaggle_reqs.txt
!pip install -q -r /tmp/kaggle_reqs.txt

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(i, torch.cuda.get_device_name(i))


Cloning into 'pixels-to-macros'...
remote: Enumerating objects: 16768, done.
remote: Counting objects: 100% (752/752), done.
remote: Compressing objects: 100% (641/641), done.
remote: Total 16768 (delta 238), reused 590 (delta 108), pack-reused 16016 (from 4)
Receiving objects: 100% (16768/16768), 1.55 GiB | 41.94 MiB/s, done.
Resolving deltas: 100% (1072/1072), done.
/kaggle/working/pixels-to-macros
Fetching origin
Updating files: 100% (14783/14783), done.
HEAD is now at 34e2d0b3 Add language filtering to recipes, fix seafood/vegetarian terms, fix macros
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.0 MB/s eta 0:00:00
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
# Cell 2 - Auto-Detect FoodSeg103 and Set Up
from pathlib import Path

# Use the exact path shown in your sidebar
DATA_DIR = Path('/kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103')

# Verify it exists
assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR}. Check the sidebar!"

OUTPUT_DIR = Path('/kaggle/working/pixels-to-macros-segformer-103')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Success! Data found at: {DATA_DIR}")
print(f"🚀 Ready to start training!")

✅ Success! Data found at: /kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103
🚀 Ready to start training!


In [3]:
# Cell 3 - Train SegFormer-B2 on FoodSeg103 (Fresh Start)
from pathlib import Path

MODEL       = 'nvidia/segformer-b2-finetuned-ade-512-512'
EPOCHS      = 80
BATCH_SIZE  = 8   # If Kaggle throws a CUDA OutOfMemory error, drop this to 4
IMG_SIZE    = 512
LR          = 6e-5
NUM_WORKERS = 2
MULTIPLIER  = 3
VAL_EVERY   = 3

!python training/train.py \
  --data_dir       {DATA_DIR} \
  --output_dir     {OUTPUT_DIR} \
  --model_name     {MODEL} \
  --num_classes    104 \
  --epochs         {EPOCHS} \
  --batch_size     {BATCH_SIZE} \
  --img_size       {IMG_SIZE} \
  --lr             {LR} \
  --num_workers    {NUM_WORKERS} \
  --val_every      {VAL_EVERY} \
  --virtual_train_multiplier {MULTIPLIER} \
  --save_every_secs 300

usage: train.py [-h] --data-dir DATA_DIR [--output-dir OUTPUT_DIR]
                [--model-name MODEL_NAME] [--num-labels NUM_LABELS]
                [--img-size IMG_SIZE] [--epochs EPOCHS]
                [--batch-size BATCH_SIZE] [--workers WORKERS] [--lr LR]
                [--weight-decay WEIGHT_DECAY] [--warmup-ratio WARMUP_RATIO]
                [--grad-clip GRAD_CLIP]
                [--checkpoint-seconds CHECKPOINT_SECONDS]
                [--val-every VAL_EVERY]
                [--virtual-train-multiplier VIRTUAL_TRAIN_MULTIPLIER]
                [--seed SEED] [--resume RESUME] [--amp] [--no-amp]
                [--no-data-parallel]
train.py: error: unrecognized arguments: --model_name nvidia/segformer-b2-finetuned-ade-512-512


In [4]:
# Cell 4 - Inspect and zip outputs
!ls -lh /kaggle/working/pixels-to-macros-segformer-154
!tail -n 20 /kaggle/working/pixels-to-macros-segformer-154/metrics.json || true
!cd /kaggle/working && zip -r pixels-to-macros-segformer-154.zip pixels-to-macros-segformer-154

ls: cannot access '/kaggle/working/pixels-to-macros-segformer-154': No such file or directory
tail: cannot open '/kaggle/working/pixels-to-macros-segformer-154/metrics.json' for reading: No such file or directory
	zip warning: name not matched: pixels-to-macros-segformer-154

zip error: Nothing to do! (try: zip -r pixels-to-macros-segformer-154.zip . -i pixels-to-macros-segformer-154)
